<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/Neural-Recommendation-Personalization-Engine/blob/main/01_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import os
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

rm: cannot remove '/content/drive/.shortcut-targets-by-id': Operation canceled
rm: cannot remove '/content/drive/MyDrive': Operation canceled
rm: cannot remove '/content/drive/.Trash-0': Directory not empty
rm: cannot remove '/content/drive/.Encrypted/.shortcut-targets-by-id': Operation canceled
rm: cannot remove '/content/drive/.Encrypted/MyDrive': Operation canceled
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
!pip install -q huggingface_hub
from huggingface_hub import login,HfApi
login()

In [3]:
# !pip install -q pandas==2.2.3
# !pip -q install -U huggingface_hub pandas pyarrow
# !pip install datasets faiss-cpu polars -q

In [12]:
!pip install -U datasets huggingface_hub

In [5]:
url = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/benchmark/5core/rating_only/Electronics.csv"
reviews = pd.read_csv(url)

print("Shape:", reviews.shape)
reviews.head()

Shape: (15473536, 4)


,user_id,parent_asin,rating,timestamp
0,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B0047T79VS,3.0,1344406083000
1,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01HHURN3W,3.0,1408995743000
2,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B00L0YLRUW,1.0,1439226089000
3,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B017T99JPG,5.0,1456772365000
4,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01LW71IBJ,5.0,1456772571000


In [6]:
print('Shape:',reviews.shape,'\n\n')
print('Columns:')
print(reviews.columns.tolist(),'\n')
print('Data Types:',reviews.dtypes,'\n')
print('Missing Values:',reviews.isna().sum())
print('\nDuplicate Rows:',reviews.duplicated().sum())
print('\n Rating Distribution',reviews['rating'].value_counts().sort_index())

print("\nUnique Users:", reviews["user_id"].nunique())
print("Unique Products:", reviews["parent_asin"].nunique())

display(reviews.head())

Shape: (15473536, 4) 


Columns:
['user_id', 'parent_asin', 'rating', 'timestamp'] 

Data Types: user_id         object
parent_asin     object
rating         float64
timestamp        int64
dtype: object 

Missing Values: user_id        0
parent_asin    0
rating         0
timestamp      0
dtype: int64

Duplicate Rows: 0

 Rating Distribution rating
1.0     1287788
2.0      713558
3.0     1074820
4.0     2190347
5.0    10207023
Name: count, dtype: int64

Unique Users: 1641026
Unique Products: 368228


,user_id,parent_asin,rating,timestamp
0,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B0047T79VS,3.0,1344406083000
1,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01HHURN3W,3.0,1408995743000
2,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B00L0YLRUW,1.0,1439226089000
3,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B017T99JPG,5.0,1456772365000
4,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01LW71IBJ,5.0,1456772571000


In [2]:
meta1 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00000-of-00010.parquet")

meta2 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00001-of-00010.parquet")

meta3 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00002-of-00010.parquet")

meta4 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00003-of-00010.parquet")

meta5 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00004-of-00010.parquet")

meta6 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00005-of-00010.parquet")

meta7 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00006-of-00010.parquet")

meta8 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00007-of-00010.parquet")

meta9 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00008-of-00010.parquet")

meta10 = pd.read_parquet("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/full-00009-of-00010.parquet")

In [3]:
for df in [meta1, meta2, meta3, meta4, meta5,
           meta6, meta7, meta8, meta9, meta10]:

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.dtypes)
    print(df.head())
    print("-" * 80)

Shape: (161002, 16)
Columns: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']
main_category       object
title               object
average_rating     float64
rating_number        int64
features            object
description         object
price               object
images              object
videos              object
store               object
categories          object
details             object
parent_asin         object
bought_together     object
subtitle            object
author              object
dtype: object
               main_category  \
0            All Electronics   
1            All Electronics   
2                  Computers   
3             AMAZON FASHION   
4  Cell Phones & Accessories   

                                               title  average_rating  \
0             FS-1051 FATSHARK TELEPORTER V3 HEADSET

In [4]:
for df in [meta1, meta2, meta3, meta4, meta5,
           meta6, meta7, meta8, meta9, meta10]:

    print(df.isna().sum())
    print("-" * 80)

main_category        1642
title                   0
average_rating          0
rating_number           0
features                0
description             0
price                   0
images                  0
videos                  0
store                 890
categories              0
details                 0
parent_asin             0
bought_together    161002
subtitle           160940
author             160965
dtype: int64
--------------------------------------------------------------------------------
main_category        1351
title                   0
average_rating          0
rating_number           0
features                0
description             0
price                   0
images                  0
videos                  0
store                 888
categories              0
details                 0
parent_asin             0
bought_together    161002
subtitle           160906
author             160957
dtype: int64
-------------------------------------------------------------

In [5]:
meta1.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,None,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Fat Shark,"[Electronics, Television & Video, Video Glasses]","{""Date First Available"": ""August 2, 2014"", ""Ma...",B00MCW7G9M,None,None,None
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",SIIG,"[Electronics, Television & Video, Accessories,...","{""Product Dimensions"": ""0.83 x 4.17 x 2.05 inc...",B00YT6XQSE,None,None,None
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],19.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['AL 2Sides Video', 'MacBook Protect...",Digi-Tatoo,"[Electronics, Computers & Accessories, Laptop ...","{""Brand"": ""Digi-Tatoo"", ""Color"": ""Fresh Marble...",B07SM135LS,None,None,None
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,[☛NotoCity 22mm band is designed for Vivoactiv...,[],9.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",NotoCity,"[Electronics, Wearable Technology, Clips, Arm ...","{""Date First Available"": ""May 29, 2020"", ""Manu...",B089CNGZCW,None,None,None
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,14.99,"{'hi_res': [None, None, None, None, None], 'la...","{'title': [], 'url': [], 'user_id': []}",Verizon,"[Electronics, Computers & Accessories, Compute...","{""Product Dimensions"": ""11.6 x 6.9 x 3.1 inche...",B004E2Z88O,None,None,None


In [14]:
metadata_dfs = [meta1, meta2, meta3, meta4, meta5,meta6, meta7, meta8, meta9, meta10]

# drop_cols = ["images","videos","bought_together","subtitle","author",'price']
drop_cols = ['price']
for df in metadata_dfs:
    df.drop(columns=drop_cols, inplace=True)

In [15]:
for i, df in enumerate(metadata_dfs, start=1):
    temp_df = df.astype(str)

    print(f"Meta{i} duplicate rows:", temp_df.duplicated().sum())

Meta1 duplicate rows: 0
Meta2 duplicate rows: 0
Meta3 duplicate rows: 0
Meta4 duplicate rows: 0
Meta5 duplicate rows: 0
Meta6 duplicate rows: 0
Meta7 duplicate rows: 0
Meta8 duplicate rows: 0
Meta9 duplicate rows: 0
Meta10 duplicate rows: 0


In [16]:
import os

os.makedirs("/content/metadata_parquet", exist_ok=True)

for i, df in enumerate(metadata_dfs, 1):
    path = f"/content/metadata_parquet/meta{i}.parquet"
    df.to_parquet(path, index=False)
    print("Saved:", path)

Saved: /content/metadata_parquet/meta1.parquet
Saved: /content/metadata_parquet/meta2.parquet
Saved: /content/metadata_parquet/meta3.parquet
Saved: /content/metadata_parquet/meta4.parquet
Saved: /content/metadata_parquet/meta5.parquet
Saved: /content/metadata_parquet/meta6.parquet
Saved: /content/metadata_parquet/meta7.parquet
Saved: /content/metadata_parquet/meta8.parquet
Saved: /content/metadata_parquet/meta9.parquet
Saved: /content/metadata_parquet/meta10.parquet


In [17]:
api = HfApi()
repo_id = "Satyamkarn100/NeuraRec-Electronics"
for i in range(1, 11):
    local_path = f"/content/metadata_parquet/meta{i}.parquet"
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=f"raw/meta{i}.parquet",
        repo_id=repo_id,
        repo_type="dataset",
        commit_message=f"Upload cleaned metadata part {i}")

    print(f"Uploaded meta{i}.parquet")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ata_parquet/meta1.parquet:  11%|#         | 15.9MB /  147MB            

Uploaded meta1.parquet


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Uploaded meta9.parquet


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ta_parquet/meta10.parquet:  13%|#2        | 16.0MB /  126MB            

Uploaded meta10.parquet
